# Lesson 20: Extending the Product

## Adding a Proofreading Agent

In this final lesson, we'll walk through a real scenario: **adding a 5th agent to the pipeline** — a proofreading agent that checks grammar, readability, and SEO best practices before the article is saved.

We won't actually run Claude Code. Instead, we'll trace through each step of the development workflow from Lesson 19, and you'll **verify the output** using everything you've learned.

This is how you'll work in practice:
1. You describe what you want
2. Claude Code implements it
3. You verify the result using your knowledge

## Step 1: Understand

Before making any change, Claude Code explores the codebase. Here's the prompt you'd give:

> "I want to add a proofreading agent to the pipeline. Before making any changes, explore the codebase and explain: (1) where the pipeline steps are defined, (2) where new agents should be added, (3) what schema pattern to follow."

Claude Code would read:
- `output/pipeline.py` — to understand the step sequence
- `output/agents/schemas.py` — to see how schemas are defined
- `output/agents/` — to see how agents are organized (one file per agent)

Let's do the same — read the key files ourselves:

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../output"))

# Read the pipeline to understand where a new step would go
pipeline_path = os.path.abspath("../../output/pipeline.py")
with open(pipeline_path, "r", encoding="utf-8") as f:
    pipeline_code = f.read()

print(f"pipeline.py ({len(pipeline_code.splitlines())} lines)\n")
print(pipeline_code)

In [ ]:
agents_path = os.path.abspath("../../output/agents/schemas.py")
with open(agents_path, "r", encoding="utf-8") as f:
    agents_code = f.read()

print(f"agents/schemas.py ({len(agents_code.splitlines())} lines)\n")
print(agents_code)

## Step 2: Plan

After understanding the codebase, you'd tell Claude Code:

> "Plan adding a proofreading agent between the writer and image agents. The proofreader should check grammar, readability, keyword usage, and SEO best practices. Return a structured result with the corrected article and a list of issues found. Use Claude Sonnet since it needs structured output."

A good plan would include:
1. **New schema** in `agents/schemas.py`: `ProofreadIssue` + `ProofreadResult`
2. **New agent** in `agents/proofreader.py`: `proofreader_agent` using Claude Sonnet + output_schema
3. **New pipeline step** in `pipeline.py`: between "writing" and "enriching"
4. **New Airtable status**: "proofreading" between "writing" and "enriching"

Let's build and verify each piece:

## Step 3: Implementation — Schema

The first thing Claude Code would create is the schema. Following the pattern from `agents/schemas.py` (and what you learned in Lessons 10 and 12), the proofreading schema would look like this:

In [ ]:
from pydantic import BaseModel, Field

# Issue found during proofreading (nested model — same pattern as OutlineSection)
class ProofreadIssue(BaseModel):
    issue_type: str = Field(description="Type: grammar, readability, seo, factual")
    location: str = Field(description="Which section or paragraph has the issue")
    description: str = Field(description="What the issue is")
    suggestion: str = Field(description="How to fix it")
    severity: str = Field(default="medium", description="low, medium, or high")

# Full proofreading result (same pattern as ContentOutline)
class ProofreadResult(BaseModel):
    corrected_article: str = Field(description="The full article with corrections applied")
    issues_found: list[ProofreadIssue] = Field(description="List of issues found and fixed")
    readability_score: int = Field(description="Readability score 1-10 (10 = easiest to read)")
    seo_score: int = Field(description="SEO optimization score 1-10")
    word_count: int = Field(description="Word count of corrected article")

# Verify the schema works (no API call — just Pydantic validation)
test_result = ProofreadResult(
    corrected_article="# Test Article\n\nThis is a corrected article.",
    issues_found=[
        ProofreadIssue(
            issue_type="grammar",
            location="Paragraph 2",
            description="Run-on sentence",
            suggestion="Split into two sentences",
            severity="medium",
        ),
        ProofreadIssue(
            issue_type="seo",
            location="Introduction",
            description="Target keyword missing from first paragraph",
            suggestion="Add 'on-page SEO' to the opening sentence",
            severity="high",
        ),
    ],
    readability_score=7,
    seo_score=8,
    word_count=150,
)

print("Schema validation passed!\n")
print(f"Corrected article: {test_result.corrected_article[:50]}...")
print(f"Issues found: {len(test_result.issues_found)}")
for issue in test_result.issues_found:
    print(f"  [{issue.severity.upper()}] {issue.issue_type}: {issue.description}")
print(f"Readability: {test_result.readability_score}/10")
print(f"SEO score: {test_result.seo_score}/10")
print(f"Word count: {test_result.word_count}")

## Step 3: Implementation — Agent

Next, Claude Code would add the agent as a new file `agents/proofreader.py`. Following the existing pattern:

In [ ]:
# This is what Claude Code would add as agents/proofreader.py
# (We're not actually running it — just reviewing the code)

proofreader_code = '''
# Proofreader Agent -- checks grammar, readability, SEO best practices
#    Model: Claude Sonnet | Tools: none | Output: ProofreadResult schema
proofreader_agent = Agent(
    name="Proofreader Agent",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    output_schema=ProofreadResult,
    instructions=[
        "You are an expert content editor and SEO proofreader.",
        "Given a Markdown article and its target keywords, proofread the article thoroughly.",
        "Check for: grammar errors, awkward phrasing, readability issues, "
        "keyword usage and density, SEO best practices (title, headings, meta description).",
        "Fix all issues directly in the article text.",
        "Return the corrected article along with a list of every issue you found and fixed.",
        "Rate readability (1-10) and SEO optimization (1-10).",
    ],
)
'''

print("Agent definition that Claude Code would generate:")
print("=" * 60)
print(proofreader_code)

## Step 3: Implementation — Pipeline Step

Finally, Claude Code would add a new step in `pipeline.py` between writing and enriching:

```python
# In pipeline.py, after the writing step:

# --- Step 4: Proofread ---
update_article_status(article_id, "proofreading")
proofread_response = proofreader_agent.run(
    f"Proofread this article. Target keywords: {keywords}\n\n{article_markdown}"
)
proofread_result = proofread_response.content
article_markdown = proofread_result.corrected_article
# Save proofread data to database...
```

The pipeline flow becomes:
```
queued → researching → outlining → writing → proofreading → enriching → review
```

## Step 4: Verify

This is where YOUR knowledge matters. Use everything you've learned to verify the implementation:

In [ ]:
# Verification checklist — applying knowledge from all modules

checks = [
    ("Module 2 (Lesson 5)", "Does the agent have a knowledge cutoff issue?",
     "No — proofreading only needs the article text, not real-time info. No tools needed."),
    
    ("Module 2 (Lesson 6)", "Are the instructions well-structured?",
     "Yes — Role ('expert editor'), Task ('proofread'), Constraints ('check grammar, SEO...')."),
    
    ("Module 2 (Lesson 7)", "Is the model choice correct?",
     "Yes — Claude Sonnet. Needs output_schema (ProofreadResult). Doesn't need tools. Grok can't do schema, so Claude is correct."),
    
    ("Module 3 (Lesson 10)", "Is the schema pattern correct?",
     "Yes — ProofreadIssue is nested inside ProofreadResult, same as OutlineSection inside ContentOutline."),
    
    ("Module 3 (Lesson 12)", "Does the nested data access work?",
     "Yes — result.issues_found[0].description works. Same pattern as outline.sections[0].heading."),
    
    ("Module 4 (Lesson 15)", "Does the pipeline step follow the pattern?",
     "Yes — update status, run agent, save results. Same as other steps."),
    
    ("Module 5 (Lesson 16)", "Are Airtable status updates correct?",
     "Need to add 'proofreading' as a valid status. Check that status flow is updated."),
    
    ("Module 5 (Lesson 17)", "Does the architecture still make sense?",
     "Yes — the new agent fits into the pipeline.py step sequence. No changes needed to chat.py or tools/workspace.py."),
]

print("=== Verification Checklist ===\n")
all_pass = True
for module, question, answer in checks:
    print(f"  [{module}]")
    print(f"  Q: {question}")
    print(f"  A: {answer}")
    print()

## More Extension Ideas

Here are more things you could ask Claude Code to build, ranked by difficulty:

| Extension | Difficulty | Claude Code prompt |
|-----------|-----------|-------------------|
| Change writing tone | Easy | "Update writer_agent instructions in agents/writer.py to write in a casual, friendly tone" |
| Add keyword density check | Medium | "Add a keyword_density field to ProofreadResult that calculates how often target keywords appear" |
| Add translation agent | Medium | "Add a 6th pipeline agent as agents/translator.py that translates the finished article to Vietnamese using Claude Sonnet" |
| Add Google Search Console tool | Hard | "Create a new GSC toolkit in output/tools/ following the Toolkit pattern in agents/image.py, then add it to the research agent" |
| Add web dashboard | Hard | "Create a simple Flask web dashboard that shows all articles and their statuses from Airtable" |

Each of these follows the same workflow: Understand -> Plan -> Implement -> Verify.

> **Already built in:** The product supports **parallel batch processing** out of the box. In chat, just say "Create articles about Topic 1, Topic 2, Topic 3 in parallel". This uses Python's `ThreadPoolExecutor` — the same concept as having multiple factory workers instead of one.

## The Development Cycle

```
  ┌─────────┐
  │  Idea   │  "I want to add proofreading"
  └────┬────┘
       ▼
  ┌─────────┐
  │  Plan   │  Claude Code explores, you approve the approach
  └────┬────┘
       ▼
  ┌──────────┐
  │Implement │  Claude Code writes the code
  └────┬─────┘
       ▼
  ┌─────────┐
  │ Verify  │  YOU check using your knowledge
  └────┬────┘
       ▼
  ┌─────────┐
  │ Deploy  │  Test it, use it, iterate
  └────┬────┘
       │
       └──────→ (back to Idea for the next improvement)
```

**You are quality control. Claude Code is the builder. Your understanding is what makes you effective.**

Without the knowledge from this course, you couldn't verify if the model choice is right, if the schema follows the pattern, or if the pipeline step is correctly placed. Now you can.

## Congratulations! You've completed all 20 lessons!

Here's the full journey:

### Module 1: Python Basics (Lessons 1-4)
Variables, lists, dictionaries, functions, packages — the foundation for reading code.

### Module 2: Understanding AI (Lessons 5-7)
How LLMs work, prompts and context, model choices — the "why" behind everything.

### Module 3: Building Agents (Lessons 8-12)
First agent, tools, structured output, chaining, mini pipeline — hands-on agent building.

### Module 4: The Real SEO Pipeline (Lessons 13-15)
Research, outline, writer, image agents — the production pipeline with real code.

### Module 5: The Complete Product (Lessons 16-18)
Airtable database, how everything connects, chat interface — turning a pipeline into a usable product.

### Module 6: AI-Assisted Development (Lessons 19-20)
Claude Code, extending the product — using AI to build with AI.

---

### Lesson-to-Production-File Mapping (Updated)

| Lessons | Concepts | Production file |
|---------|----------|-----------------|
| 05-07 | LLMs, prompts, models | (Conceptual — informs all decisions) |
| 08-09 | Agent creation, tools | `output/agents/researcher.py`, `output/agents/outliner.py`, `output/agents/writer.py`, `output/agents/image.py` |
| 10, 13 | Pydantic schemas | `output/agents/schemas.py` |
| 11-12, 13-15 | Agent chaining, pipeline | `output/pipeline.py` |
| 09, 14 | Image toolkits | `output/agents/image.py` |
| 16 | Database (Airtable) | `output/tools/airtable.py` |
| 17 | How everything connects | All files — the full request flow |
| 18 | Chat interface, workspace tools | `output/chat.py`, `output/tools/workspace.py` |
| 19-20 | AI-assisted development | `CLAUDE.md` (the blueprint for Claude Code) |

---

### What's Next?

1. **Start using the product** — run `python output/chat.py` and start creating articles for your team
2. **Customize agent instructions** — tweak the writing style, tone, SEO approach
3. **Try Claude Code** — install it and make a small change (e.g., change the writer's tone)
4. **Build your own tools** — add tools your SEO team needs (GSC, Analytics, etc.)
5. **Teach others** — you now understand enough to help your colleagues

**The best way to learn is to build. Start small, verify carefully, iterate quickly.**

Good luck!

In [ ]:
# Final exercise: Explore the full project structure

import os

output_dir = os.path.abspath("../../output")
print("Your production codebase:\n")

for root, dirs, files in os.walk(output_dir):
    # Skip __pycache__
    dirs[:] = [d for d in dirs if d != "__pycache__"]
    level = root.replace(output_dir, "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root)
    print(f"{indent}{folder_name}/")
    sub_indent = "  " * (level + 1)
    for file in sorted(files):
        if file.endswith(".py"):
            filepath = os.path.join(root, file)
            with open(filepath, "r", encoding="utf-8") as f:
                lines = len(f.readlines())
            print(f"{sub_indent}{file} ({lines} lines)")

print("\nYou understand every one of these files.")
print("That knowledge is your superpower when working with Claude Code.")